#Rank Zero-shot MGT Detector

##GPT2-XL Backbone

In [ ]:
import os, sys, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "gpt2-xl"
print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# RANK SCORE

def compute_rank_score(text, model, tokenizer, device, max_len=512):

    try:
        inp = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=False
        ).to(device)

        ids = inp["input_ids"]
        if ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            logits = model(ids).logits

        ranks = []
        for i in range(ids.shape[1] - 1):
            pos_logits = logits[0, i, :]
            target_token = ids[0, i + 1].item()
            rank = (pos_logits > pos_logits[target_token]).sum().item() + 1
            ranks.append(rank)

        return np.mean(ranks) if ranks else np.nan
    except:
        return np.nan

# PREPARE FUNCTION (handles LLM-edited columns)

def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER

def evaluate_rank(texts, labels, model, tokenizer, device):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_rank_score(t, model, tokenizer, device)
        scores.append(50000.0 if (np.isnan(s) or np.isinf(s)) else s)
    scores = [-s for s in scores]

    thr = np.median([s for s in scores if np.isfinite(s)])
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET

print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")

tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}

# RUN ALL TASKS

PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_rank(X, y, model, tokenizer, device)
    res[name] = mt

    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_rank_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")


# SUMMARY

bn = BACKBONE.replace("/", "_")
summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/rank_{bn}_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY ")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")

Using gpt2-xl on cuda


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

AUROC: 0.5077 | Acc: 0.4965 | F1: 0.5972

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [07:54<00:00, 18.42it/s]


AUROC: 0.5665 | Acc: 0.5360 | F1: 0.3813

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [07:23<00:00, 19.73it/s]


AUROC: 0.5377 | Acc: 0.5195 | F1: 0.6156

 FINAL SUMMARY 
         auroc  accuracy      f1
H_vs_M  0.6033    0.5711  0.5711
E_vs_M  0.5636    0.5437  0.5437
H_vs_E  0.5476    0.5409  0.5409
L_vs_M  0.5973    0.5543  0.4057
H_vs_L  0.5077    0.4965  0.5972
G_vs_M  0.5665    0.5360  0.3813
H_vs_G  0.5377    0.5195  0.6156

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/


##OPT-1.3B Backbone

In [ ]:
import os, sys, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "facebook/opt-1.3b"
if device == "cpu":
    BACKBONE = "gpt2-xl"
print(f"Using {BACKBONE} on {device}")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# RANK SCORE

def compute_rank_score(text, model, tokenizer, device, max_len=512):

    try:
        inp = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=False
        ).to(device)

        ids = inp["input_ids"]
        if ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            logits = model(ids).logits

        ranks = []
        for i in range(ids.shape[1] - 1):
            pos_logits = logits[0, i, :]
            target_token = ids[0, i + 1].item()
            rank = (pos_logits > pos_logits[target_token]).sum().item() + 1
            ranks.append(rank)

        return np.mean(ranks) if ranks else np.nan
    except:
        return np.nan

# PREPARE FUNCTION (handles LLM-edited columns)

def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER

def evaluate_rank(texts, labels, model, tokenizer, device):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_rank_score(t, model, tokenizer, device)
        scores.append(50000.0 if (np.isnan(s) or np.isinf(s)) else s)
    scores = [-s for s in scores]
    thr = np.median([s for s in scores if np.isfinite(s)])
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET

print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")

tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}

# RUN ALL TASKS

PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_rank(X, y, model, tokenizer, device)
    res[name] = mt

    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_rank_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")


# SUMMARY

bn = BACKBONE.replace("/", "_")
summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/rank_{bn}_summary.csv")

print("\n" + "="*60)
print(" FINAL SUMMARY (OPT-1.3B)")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n Results saved to {PROJECT}/results/")

Using facebook/opt-1.3b on cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

...

AUROC: 0.4753 | Acc: 0.4738 | F1: 0.5791

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [05:28<00:00, 26.65it/s]


AUROC: 0.5691 | Acc: 0.5330 | F1: 0.3774

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [05:03<00:00, 28.79it/s]


AUROC: 0.5484 | Acc: 0.5257 | F1: 0.6206

 FINAL SUMMARY (OPT-1.3B)
         auroc  accuracy      f1
H_vs_M  0.6154    0.5812  0.5812
E_vs_M  0.5700    0.5409  0.5409
H_vs_E  0.5519    0.5373  0.5373
L_vs_M  0.6343    0.5762  0.4350
H_vs_L  0.4753    0.4738  0.5791
G_vs_M  0.5691    0.5330  0.3774
H_vs_G  0.5484    0.5257  0.6206

 Results saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/


##Falcon-7B Backbone

In [ ]:
import os, sys, torch, numpy as np, pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from datasets import load_dataset
import ast

# MODEL SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "tiiuae/falcon-7b"

print(f"Using {BACKBONE} on {device}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB" if device == "cuda" else "CPU mode")

model = AutoModelForCausalLM.from_pretrained(
    BACKBONE,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    low_cpu_mem_usage=True,
    trust_remote_code=True
).eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# RANK SCORE

def compute_rank_score(text, model, tokenizer, device, max_len=512):
    """
    Compute average rank score (not log).
    """
    try:
        inp = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_len,
            padding=False
        ).to(device)

        ids = inp["input_ids"]
        if ids.shape[1] < 2:
            return np.nan

        with torch.no_grad():
            logits = model(ids).logits

        ranks = []
        for i in range(ids.shape[1] - 1):
            pos_logits = logits[0, i, :]
            target_token = ids[0, i + 1].item()
            rank = (pos_logits > pos_logits[target_token]).sum().item() + 1
            ranks.append(rank)

        return np.mean(ranks) if ranks else np.nan
    except:
        return np.nan

# PREPARE FUNCTION

def prepare(df, col1, col2):

    X, y = [], []

    for _, r in df.iterrows():
        for col, lab in [(col1, 0), (col2, 1)]:
            val = r.get(col, None)

            if pd.isna(val):
                continue

            if isinstance(val, str) and val.strip().startswith('[{'):
                try:
                    parsed = ast.literal_eval(val)
                    for item in parsed:
                        if isinstance(item, dict):
                            for prompt_key in ['P1', 'P2', 'P3']:
                                if prompt_key in item:
                                    text = str(item[prompt_key]).strip()
                                    if text:
                                        X.append(text)
                                        y.append(lab)
                                    break
                except:
                    text = str(val).strip()
                    if text:
                        X.append(text)
                        y.append(lab)
            else:
                text = str(val).strip()
                if text:
                    X.append(text)
                    y.append(lab)

    return X, y

# EVALUATION HELPER

def evaluate_rank(texts, labels, model, tokenizer, device):
    scores = []
    for t in tqdm(texts, desc="Scoring", ncols=80):
        s = compute_rank_score(t, model, tokenizer, device)
        scores.append(50000.0 if (np.isnan(s) or np.isinf(s)) else s)
    scores = [-s for s in scores]

    thr = np.median([s for s in scores if np.isfinite(s)])
    preds = [1 if s > thr else 0 for s in scores]

    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    return scores, preds, {
        'accuracy': acc,
        'auroc': auroc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'threshold': thr
    }

# LOAD BEEMO DATASET

print("\nLoading BEEMO dataset...")
df = load_dataset("toloka/beemo")['train'].to_pandas()
print(f"Loaded {len(df)} samples")

tasks = {
    'H_vs_M': ('human_output', 'model_output'),
    'E_vs_M': ('human_edits', 'model_output'),
    'H_vs_E': ('human_output', 'human_edits'),
    'L_vs_M': ('llama-3.1-70b_edits', 'model_output'),
    'H_vs_L': ('human_output', 'llama-3.1-70b_edits'),
    'G_vs_M': ('gpt-4o_edits', 'model_output'),
    'H_vs_G': ('human_output', 'gpt-4o_edits'),
}

# RUN ALL TASKS

PROJECT = "/content/drive/MyDrive/BEEMO_Reproducibility"
os.makedirs(f"{PROJECT}/results", exist_ok=True)
res = {}

for name, (c1, c2) in tasks.items():
    print(f"\n{'='*60}\n {name}\n{'='*60}")
    X, y = prepare(df, c1, c2)
    print(f"Samples: {len(X)} ({y.count(0)}/{y.count(1)})")

    sc, pr, mt = evaluate_rank(X, y, model, tokenizer, device)
    res[name] = mt

    bn = BACKBONE.replace("/", "_")
    pd.DataFrame({
        'text': X,
        'label': y,
        'score': sc,
        'pred': pr
    }).to_csv(
        f"{PROJECT}/results/{name}_rank_{bn}.csv",
        index=False
    )
    print(f"AUROC: {mt['auroc']:.4f} | Acc: {mt['accuracy']:.4f} | F1: {mt['f1']:.4f}")

# SUMMARY

bn = BACKBONE.replace("/", "_")
summary = pd.DataFrame(res).T
summary.to_csv(f"{PROJECT}/results/rank_{bn}_summary.csv")


print(" FINAL SUMMARY - RANK (Falcon-7B)")
print("="*60)
print(summary[['auroc', 'accuracy', 'f1']].round(4))
print(f"\n saved to {PROJECT}/results/")

Using tiiuae/falcon-7b on cuda
GPU Memory: 85.17 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configuration_falcon.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-7b:
- configuration_falcon.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!



...

AUROC: 0.4546 | Acc: 0.4615 | F1: 0.5692

 G_vs_M
Samples: 8748 (6561/2187)


Scoring: 100%|██████████████████████████████| 8748/8748 [08:20<00:00, 17.46it/s]


AUROC: 0.5543 | Acc: 0.5262 | F1: 0.3682

 H_vs_G
Samples: 8748 (2187/6561)


Scoring: 100%|██████████████████████████████| 8748/8748 [07:56<00:00, 18.36it/s]


AUROC: 0.5357 | Acc: 0.5207 | F1: 0.6166
 FINAL SUMMARY - RANK (Falcon-7B)
         auroc  accuracy      f1
H_vs_M  0.5884    0.5642  0.5642
E_vs_M  0.5611    0.5432  0.5432
H_vs_E  0.5315    0.5258  0.5258
L_vs_M  0.6286    0.5740  0.4319
H_vs_L  0.4546    0.4615  0.5692
G_vs_M  0.5543    0.5262  0.3682
H_vs_G  0.5357    0.5207  0.6166

 saved to /content/drive/MyDrive/BEEMO_Reproducibility/results/
